# load data

In [1]:
# Import core analytical and visualization libraries
import pandas as pd
import numpy as np
import os
import warnings

# Data visualization libraries
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import matplotlib.patches as mpatches
from matplotlib.ticker import FuncFormatter

# Ignore warning messages for cleaner notebook output
warnings.filterwarnings('ignore')

In [2]:
DATA_PATH = '../dataset/02_after_clean/'
print("Loading data...")

# Master
df_products = pd.read_parquet(DATA_PATH + 'products.parquet')
df_customers = pd.read_parquet(DATA_PATH + 'customers.parquet')
df_promotions = pd.read_parquet(DATA_PATH + 'promotions.parquet')
df_geography = pd.read_parquet(DATA_PATH + 'geography.parquet')

# Transaction
df_orders = pd.read_parquet(DATA_PATH + 'orders.parquet')
df_order_items = pd.read_parquet(DATA_PATH + 'order_items.parquet')
df_shipments = pd.read_parquet(DATA_PATH + 'shipments.parquet')
df_returns = pd.read_parquet(DATA_PATH + 'returns.parquet')
df_reviews = pd.read_parquet(DATA_PATH + 'reviews.parquet')

# Analytical
df_sales = pd.read_parquet(DATA_PATH + 'sales.parquet')
#df_sample_submission = pd.read_parquet(DATA_PATH + 'sample_submission.parquet')

# Operational
df_inventory = pd.read_parquet(DATA_PATH + 'inventory.parquet')
df_web_traffic = pd.read_parquet(DATA_PATH + 'web_traffic.parquet')

print("Load data successfully")

Loading data...
Load data successfully


In [3]:
df_order_items.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 714669 entries, 0 to 714668
Data columns (total 7 columns):
 #   Column           Non-Null Count   Dtype  
---  ------           --------------   -----  
 0   order_id         714669 non-null  string 
 1   product_id       714669 non-null  string 
 2   quantity         714669 non-null  int64  
 3   unit_price       714669 non-null  float64
 4   discount_amount  714669 non-null  float64
 5   promo_id         714669 non-null  string 
 6   promo_id_2       714669 non-null  string 
dtypes: float64(2), int64(1), string(4)
memory usage: 38.2 MB


# FE

## merge product to order_item

In [4]:
df_order_items = df_order_items.merge(df_products, on='product_id', how='left')
df_order_items = df_order_items.drop(columns=['price', 'discount_amount', 'promo_id', 'promo_id_2'])
df_order_items = df_order_items.merge(df_orders[['order_id', 'order_date']], on='order_id', how='left')
df_order_items['gross_rev'] = df_order_items['quantity'] * df_order_items['unit_price']
df_order_items['total_cogs'] = df_order_items['quantity'] * df_order_items['cogs']
df_order_items.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 714669 entries, 0 to 714668
Data columns (total 13 columns):
 #   Column        Non-Null Count   Dtype         
---  ------        --------------   -----         
 0   order_id      714669 non-null  string        
 1   product_id    714669 non-null  string        
 2   quantity      714669 non-null  int64         
 3   unit_price    714669 non-null  float64       
 4   product_name  714669 non-null  string        
 5   category      714669 non-null  category      
 6   segment       714669 non-null  category      
 7   size          714669 non-null  category      
 8   color         714669 non-null  category      
 9   cogs          714669 non-null  float64       
 10  order_date    714669 non-null  datetime64[ns]
 11  gross_rev     714669 non-null  float64       
 12  total_cogs    714669 non-null  float64       
dtypes: category(4), datetime64[ns](1), float64(4), int64(1), string(3)
memory usage: 51.8 MB


In [5]:
df_sales = df_sales.drop(columns=['Revenue', 'COGS'])
df_sales['month_day'] = df_sales['Date'].dt.strftime('%m-%d')
df_sales['day'] = df_sales['Date'].dt.day
df_sales['month'] = df_sales['Date'].dt.month
df_sales['year'] = df_sales['Date'].dt.year
df_sales['day_of_week'] = df_sales['Date'].dt.dayofweek
df_sales['is_weekend'] = df_sales['day_of_week'].apply(lambda x: 1 if x >= 5 else 0)
df_sales['is_first_7_days'] = (df_sales['day'] <= 7).astype(int)
df_sales['is_last_7_days'] = (df_sales['day'] > df_sales['Date'].dt.days_in_month - 7).astype(int)
df_sales['quarter'] = df_sales['month'] % 12 // 3 + 1

df_sales['is_Spring_promotion'] = df_sales['month_day'].between('03-18', '04-17').astype(int)
df_sales['is_MidYear_promotion'] = df_sales['month_day'].between('06-23', '07-22').astype(int)
df_sales['is_FallLauch_promotion'] = (df_sales['month_day'].between('08-30', '10-01') | ((df_sales['year'] % 4 == 1) & (df_sales['month_day'] == '10-02'))).astype(int)
df_sales['is_YearEnd_promotion'] = (df_sales['month_day'].between('11-18', '12-31') |  df_sales['month_day'].between('01-01', '01-02')).astype(int) # có nhiều ngày kết thúc khác nhưng fix vào 02/01 hàng năm
df_sales['is_Rural_promotion'] = ((df_sales['year'] % 2 == 1)  & df_sales['month_day'].between('01-30', '03-01')).astype(int) # từ năm 2015 mới cố định ngày kết thúc
df_sales['is_Urban_promotion'] = ((df_sales['year'] % 2 == 1)  & df_sales['month_day'].between('07-30', '09-02')).astype(int)
df_sales['category_promotion'] = -1 # default: if it isn't in any promotion
df_sales['chanel_promotion'] = -1 # default: if it isn't in any promotion

#decode category_promotion: 0 -> All ; 1 -> Outdoor ; 2 -> Steetwear
df_sales.loc[(df_sales['is_Spring_promotion'] | df_sales['is_MidYear_promotion'] | df_sales['is_YearEnd_promotion']), 'category_promotion'] = 0
df_sales.loc[df_sales['is_Rural_promotion'], 'category_promotion'] = 1
df_sales.loc[df_sales['is_Urban_promotion'], 'category_promotion'] = 2

#decode chanel_promotion: 0 -> all_chanel ; 1 -> in_store ; 2 -> online ; 3 -> onl or social ; 4 onl or email or social
df_sales.loc[(df_sales['is_Spring_promotion'] | df_sales['is_FallLauch_promotion']), 'chanel_promotion'] = 4
df_sales.loc[df_sales['is_Rural_promotion'], 'chanel_promotion'] = 1
df_sales.loc[df_sales['is_Urban_promotion'], 'chanel_promotion'] = 2
df_sales.loc[df_sales['is_MidYear_promotion'], 'chanel_promotion'] = 3
df_sales.loc[df_sales['is_YearEnd_promotion'], 'chanel_promotion'] = 0

df_sales = df_sales.drop(columns=['month_day', 'day', 'year'])
df_order_items.info()
df_sales.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 714669 entries, 0 to 714668
Data columns (total 13 columns):
 #   Column        Non-Null Count   Dtype         
---  ------        --------------   -----         
 0   order_id      714669 non-null  string        
 1   product_id    714669 non-null  string        
 2   quantity      714669 non-null  int64         
 3   unit_price    714669 non-null  float64       
 4   product_name  714669 non-null  string        
 5   category      714669 non-null  category      
 6   segment       714669 non-null  category      
 7   size          714669 non-null  category      
 8   color         714669 non-null  category      
 9   cogs          714669 non-null  float64       
 10  order_date    714669 non-null  datetime64[ns]
 11  gross_rev     714669 non-null  float64       
 12  total_cogs    714669 non-null  float64       
dtypes: category(4), datetime64[ns](1), float64(4), int64(1), string(3)
memory usage: 51.8 MB
<class 'pandas.core.frame.Data

In [6]:
def create_full_features(df, attribute_col, prefix):
    # Đảm bảo cột order_date chỉ lấy phần ngày
    df['date_only'] = pd.to_datetime(df['order_date']).dt.normalize()

    # 1. TÍNH TỔNG SỐ LƯỢNG TRONG NGÀY (Đây chính là biến mới bạn cần)
    # Kết quả: bảng cross-tab với index là ngày, cột là các loại, giá trị là tổng số lượng
    daily_qty = df.groupby(['date_only', attribute_col], observed=False)['quantity'].sum().unstack(fill_value=0)
    
    # Tạo biến Quantity: Giữ nguyên giá trị daily_qty và đổi tên cột
    qty_features = daily_qty.copy()
    qty_features.columns = [f'qty_daily_{prefix}_{col}' for col in qty_features.columns]

    # 2. XẾP HẠNG TRONG NGÀY
    daily_rank = daily_qty.rank(axis=1, method='min', ascending=False).astype(int)
    daily_rank.columns = [f'rank_daily_{prefix}_{col}' for col in daily_rank.columns]

    # 3. XẾP HẠNG TRONG 30 NGÀY QUA
    rolling_30d_qty = daily_qty.rolling('30D').sum()
    rolling_30d_rank = rolling_30d_qty.rank(axis=1, method='min', ascending=False).astype(int)
    rolling_30d_rank.columns = [f'rank_30d_{prefix}_{col}' for col in rolling_30d_rank.columns]

    # Ghép cả 3 nhóm biến (Quantity ngày, Rank ngày, Rank 30 ngày) lại với nhau
    return pd.concat([qty_features, daily_rank, rolling_30d_rank], axis=1)


# 1. Tạo toàn bộ features cho 4 phân loại
features_category = create_full_features(df_order_items, 'category', 'cat')
features_segment = create_full_features(df_order_items, 'segment', 'seg')
features_size = create_full_features(df_order_items, 'size', 'size')
features_color = create_full_features(df_order_items, 'color', 'color')

# 2. Ghép tất cả các ma trận lại thành 1 siêu bảng
all_features = pd.concat([features_category, features_segment, features_size, features_color], axis=1)
all_features_past = all_features.shift(1)

# 2. ĐỔI TÊN CỘT: Cập nhật tên để dễ nhận biết đây là dữ liệu quá khứ
new_col_names = []
for col in all_features_past.columns:
    # Thay chữ 'daily' thành 'lag1' (đại diện cho hôm qua)
    new_name = col.replace('daily', 'lag1')
    
    # Thêm chữ 'lag1' vào các biến 30 ngày
    new_name = new_name.replace('rank_30d', 'rank_30d_lag1')
    
    new_col_names.append(new_name)

# Gán danh sách tên mới lại cho bảng
all_features_past.columns = new_col_names

# Merge dữ liệu quá khứ vào bảng ngày
df_sales = df_sales.merge(all_features_past, left_on='Date', right_index=True, how='left')
df_sales.fillna(0, inplace=True)

In [7]:
rev_per_day = df_order_items.groupby('order_date')['gross_rev'].sum()
cog_per_day = df_order_items.groupby('order_date')['total_cogs'].sum()
df_sales['Revenue'] = df_sales['Date'].map(rev_per_day)
df_sales['COGS'] = df_sales['Date'].map(cog_per_day)

df_sales['rev_lag_1'] = df_sales['Revenue'].shift(1).fillna(0.0)
df_sales['rev_lag_7'] = df_sales['Revenue'].shift(7).fillna(0.0)
df_sales['rev_lag_365'] = df_sales['Revenue'].shift(365).fillna(0.0)

df_sales['cog_lag_1'] = df_sales['COGS'].shift(1).fillna(0.0)
df_sales['cog_lag_7'] = df_sales['COGS'].shift(7).fillna(0.0)
df_sales['cog_lag_365'] = df_sales['COGS'].shift(365).fillna(0.0)


In [8]:
EXPORT_PATH = '../dataset/04_after_fe/'
os.makedirs(EXPORT_PATH, exist_ok=True)
# Review dataset schemas and non-null counts
df_sales.info()
# Export the processed dataframes
df_sales.to_parquet(EXPORT_PATH + 'train_data.parquet', index=False)
print("Data preparation complete.")

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3833 entries, 0 to 3832
Columns: 101 entries, Date to cog_lag_365
dtypes: datetime64[ns](1), float64(86), int32(3), int64(11)
memory usage: 2.9 MB
Data preparation complete.
